# Week 4 · Lab 1 — Train your first CNN: ships in satellite images
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trongan93/dl-space-2026/blob/main/notebooks/Week4_Lab1_Ship_CNN_Training.ipynb)  ·  repo: [trongan93/dl-space-2026](https://github.com/trongan93/dl-space-2026)

**Deep Learning in Space Technology Applications · 115-1 · NTUT · Dr. Trong-An Bui**
Colab, GPU runtime (Runtime → Change runtime type → T4). About 25 minutes end to end; the training itself takes ~1 minute.

Before you use a detector as a black box (Lab 2), you build the smallest possible one yourself. The task is the oldest one in satellite computer vision: **is there a ship in this 80 × 80 pixel chip?** You will

1. load 4 000 chips cut from Planet scenes of San Francisco Bay and Long Beach (1 000 ships, 3 000 not-ships, 3 m pixels),
2. split them **by scene** (Week 3: the honest split) and build a classical baseline,
3. define a 3-convolution CNN in PyTorch, train it, watch the two-loop picture (train loss ↓, validation accuracy ↑),
4. measure it the Week 3 way — confusion matrix, precision / recall, the threshold,
5. look at what the first convolution layer learned (Week 3's kernels, now fitted to data),
6. turn the classifier into a **detector** with a sliding window over a whole scene — and see why that is slow and boxy, which is exactly the problem YOLO (Lab 2) solves.

> Hand in (with Lab 2, Sunday 11 Oct 23:59): the executed notebook as PDF + .ipynb, and the four answers in section 9.

## 0 · Setup

In [ ]:
import os, io, json, time, math, urllib.request, numpy as np, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", DEVICE, "| GPU:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "-")
if DEVICE == "cpu": print(">>> No GPU: Runtime -> Change runtime type -> T4. Training still works on CPU, just slower (~3 min).")

## 1 · The data — 4 000 chips, two classes, 434 scenes
The chips come from the *Ships in Satellite Imagery* set (Planet PlanetScope, 3 m GSD, RGB). Each chip is 80 × 80 × 3 `uint8`; label **1 = ship** (the chip is centred on a vessel), **0 = not ship** (water, land, docks, partial ships, waves).
The file name of every original chip encodes the **scene it was cut from** — we keep that, because it decides how we may split.

In [ ]:
DATA = "https://raw.githubusercontent.com/trongan93/dl-space-2026/main/data/shipsnet/"
for fn in ["shipsnet_80x80.npz", "scene_sfbay_1.jpg", "scene_lb_1.jpg"]:
    for cand in [fn, os.path.join("data", "shipsnet", fn), os.path.join("..", "data", "shipsnet", fn)]:
        if os.path.exists(cand): break
    else:
        print("downloading", fn); urllib.request.urlretrieve(DATA + fn, fn); cand = fn
    globals()[fn.split(".")[0].upper()] = cand
d = np.load(SHIPSNET_80X80, allow_pickle=False)
X, y, scene = d["X"], d["y"].astype(np.int64), d["scene"]
print("X", X.shape, X.dtype, "| y", y.shape, "| ships:", int(y.sum()), "| not-ships:", int((y == 0).sum()), "| scenes:", len(np.unique(scene)))
print("pixel value range", X.min(), "-", X.max(), "(uint8: 0-255 - this is a display product, not reflectance; note it on the card)")

In [ ]:
fig, ax = plt.subplots(2, 10, figsize=(16, 3.6))
for r, lab in enumerate([1, 0]):
    idx = np.flatnonzero(y == lab)[:10]
    for a, i in zip(ax[r], idx): a.imshow(X[i]); a.axis("off")
    ax[r, 0].set_title("ship" if lab else "not ship", loc="left", fontsize=10, color="green" if lab else "grey")
plt.suptitle("80 x 80 px chips at 3 m: a 30 m ship is ~10 pixels long - about the limit of what a small CNN can learn", y=1.02); plt.tight_layout(); plt.show()

## 2 · Split by scene, not by chip  *(Week 3, slide 'Split design')*
Chips from the same scene share the sun angle, water colour and haze. A random split would let the network recognise the *scene* rather than the *ship*. We hold out whole scenes: ~70 % of scenes train, 15 % validate, 15 % test.

In [ ]:
rng = np.random.default_rng(0)
scenes = np.unique(scene); rng.shuffle(scenes)
n = len(scenes); s_train, s_val, s_test = scenes[: int(.7 * n)], scenes[int(.7 * n): int(.85 * n)], scenes[int(.85 * n):]
part = np.where(np.isin(scene, s_train), "train", np.where(np.isin(scene, s_val), "val", "test"))
for p in ["train", "val", "test"]:
    m = part == p; print(f"{p:5s}: {m.sum():4d} chips from {len(np.unique(scene[m])):3d} scenes, ships {100 * y[m].mean():.1f} %")
# a random-chip split, only to measure the leakage gap later
part_random = rng.choice(["train", "val", "test"], size=len(y), p=[.7, .15, .15])

## 3 · Baseline first  *(course rule: no CNN result without the classical baseline beside it)*
Logistic regression on the raw 19 200 pixel values, and a random forest on eight hand-made features (mean / std per band, mean brightness of the centre vs the border). If the CNN cannot beat these, it has not earned its GPU.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
def feats(X):
    Xf = X.astype(np.float32) / 255
    centre = Xf[:, 25:55, 25:55].mean((1, 2, 3)); border = np.concatenate([Xf[:, :10].reshape(len(X), -1), Xf[:, -10:].reshape(len(X), -1)], 1).mean(1)
    return np.column_stack([Xf.mean((1, 2)), Xf.std((1, 2)), centre, border, centre - border, Xf.max((1, 2, 3))])
tr, va, te = part == "train", part == "val", part == "test"
t0 = time.time(); lr = LogisticRegression(max_iter=2000).fit(X[tr].reshape(tr.sum(), -1) / 255., y[tr])
rf = RandomForestClassifier(300, n_jobs=-1, random_state=0).fit(feats(X[tr]), y[tr])
def report(name, pred, ref):
    p, r, f, _ = precision_recall_fscore_support(ref, pred, average="binary"); print(f"  {name:32s} acc {accuracy_score(ref, pred):.3f}  precision {p:.3f}  recall {r:.3f}  F1 {f:.3f}")
print("TEST scenes (never seen):")
report("logistic regression on pixels", lr.predict(X[te].reshape(te.sum(), -1) / 255.), y[te])
report("random forest on 8 features", rf.predict(feats(X[te])), y[te])
print("baselines fitted in %.0f s" % (time.time() - t0))

## 4 · The CNN — three convolutions, one decision  *(slides 6, 12, 16)*
The same shape as the 'backbone → head' picture: three `Conv → ReLU → MaxPool` blocks turn 80 × 80 × 3 into 10 × 10 × 64 feature maps; a small dense head turns those into two scores. About 430 000 parameters — six times smaller than YOLO26n, and most of them sit in the dense head.

In [ ]:
class ShipNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),     # 80 -> 40
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),    # 40 -> 20
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))    # 20 -> 10
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(64 * 10 * 10, 64), nn.ReLU(), nn.Linear(64, 2))
    def forward(self, x): return self.head(self.backbone(x))
net = ShipNet().to(DEVICE)
print(net); print("parameters: %.0f k" % (sum(p.numel() for p in net.parameters()) / 1e3))
x = torch.zeros(1, 3, 80, 80, device=DEVICE)
for layer in net.backbone: x = layer(x); print(f"{layer.__class__.__name__:10s} -> {tuple(x.shape)}") if not isinstance(layer, nn.ReLU) else None

### 4b · Tensors, normalisation, augmentation  *(Week 3, slide 'Normalisation and augmentation')*
`(N, 80, 80, 3) uint8` → `(N, 3, 80, 80) float32`, standardised with the **training-set** mean and std per band. Flips and 90° rotations are free for nadir imagery, so we add them to the training loader only.

In [ ]:
MEAN = (X[tr].astype(np.float32) / 255).mean((0, 1, 2)); STD = (X[tr].astype(np.float32) / 255).std((0, 1, 2))
print("train mean per band", MEAN.round(3), "| std", STD.round(3), " <- these six numbers go on the dataset card")
def to_tensor(Xn): return torch.tensor(((Xn.astype(np.float32) / 255 - MEAN) / STD).transpose(0, 3, 1, 2))
class Chips(torch.utils.data.Dataset):
    def __init__(self, Xn, yn, augment=False): self.x, self.y, self.aug = to_tensor(Xn), torch.tensor(yn), augment
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        x = self.x[i]
        if self.aug:
            if torch.rand(1) < .5: x = x.flip(-1)
            if torch.rand(1) < .5: x = x.flip(-2)
            x = torch.rot90(x, int(torch.randint(0, 4, (1,))), (-2, -1))
        return x, self.y[i]
mk = lambda m, aug=False, bs=64, sh=False: torch.utils.data.DataLoader(Chips(X[m], y[m], aug), batch_size=bs, shuffle=sh)
train_dl, val_dl, test_dl = mk(tr, True, 64, True), mk(va), mk(te)
xb, yb = next(iter(train_dl)); print("one batch:", tuple(xb.shape), xb.dtype, "| labels", tuple(yb.shape))

## 5 · Train — the two loops  *(Week 3, slide 'Train / validation / test')*
Inner loop: every training batch → forward → loss → backward → optimiser step. Outer loop: after each epoch, score the **validation** set and keep the best weights. The test set is touched once, at the end.

In [ ]:
def evaluate(model, dl):
    model.eval(); correct = 0; loss = 0; probs = []
    with torch.no_grad():
        for xb, yb in dl:
            out = model(xb.to(DEVICE)); loss += F.cross_entropy(out, yb.to(DEVICE), reduction="sum").item()
            probs.append(F.softmax(out, 1)[:, 1].cpu()); correct += (out.argmax(1).cpu() == yb).sum().item()
    return loss / len(dl.dataset), correct / len(dl.dataset), torch.cat(probs).numpy()

EPOCHS = 12
net = ShipNet().to(DEVICE); opt = torch.optim.Adam(net.parameters(), lr=1e-3)
hist = []; best = (0, None); t0 = time.time()
for ep in range(1, EPOCHS + 1):
    net.train(); run = 0
    for xb, yb in train_dl:                                   # inner loop
        opt.zero_grad(); loss = F.cross_entropy(net(xb.to(DEVICE)), yb.to(DEVICE)); loss.backward(); opt.step(); run += loss.item() * len(yb)
    tl = run / len(train_dl.dataset); vl, va_acc, _ = evaluate(net, val_dl)   # outer loop: validation decides
    hist.append((tl, vl, va_acc))
    if va_acc > best[0]: best = (va_acc, {k: v.clone() for k, v in net.state_dict().items()})
    print(f"epoch {ep:2d}  train loss {tl:.3f}  val loss {vl:.3f}  val acc {va_acc:.3f}  {time.time() - t0:4.0f} s")
net.load_state_dict(best[1]); print("best validation accuracy %.3f - those weights are kept" % best[0])
h = np.array(hist); fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(h[:, 0], label="train loss"); ax[0].plot(h[:, 1], label="val loss"); ax[0].legend(); ax[0].set_xlabel("epoch"); ax[0].set_title("loss: down on train; if val turns up, you are overfitting")
ax[1].plot(h[:, 2], color="green"); ax[1].set_xlabel("epoch"); ax[1].set_title("validation accuracy"); plt.tight_layout(); plt.show()

## 6 · Measure it the Week 3 way — on the test scenes
Accuracy, then the confusion matrix, then precision / recall at the default threshold 0.5, then a threshold sweep. And the **leakage gap**: the same network trained with a random-chip split.

In [ ]:
from sklearn.metrics import confusion_matrix
_, test_acc, p_test = evaluate(net, test_dl)
pred = (p_test > 0.5).astype(int)
print("TEST scenes:"); report("3-conv CNN (scene split)", pred, y[te])
cm = confusion_matrix(y[te], pred); print("confusion matrix [[TN FP] [FN TP]]:\n", cm)
print("\nthreshold sweep (precision / recall on test):")
for T in [0.1, 0.3, 0.5, 0.7, 0.9]:
    pr = (p_test > T).astype(int); p, r, f, _ = precision_recall_fscore_support(y[te], pr, average="binary"); print(f"  T = {T:.1f}   precision {p:.3f}   recall {r:.3f}   F1 {f:.3f}   ships flagged {pr.sum()}")
# the leakage gap: same recipe, random chip split
trR, teR = part_random == "train", part_random == "test"
MEAN_R, STD_R = MEAN, STD
netR = ShipNet().to(DEVICE); optR = torch.optim.Adam(netR.parameters(), lr=1e-3); dlR = mk(trR, True, 64, True)
for ep in range(EPOCHS):
    netR.train()
    for xb, yb in dlR: optR.zero_grad(); F.cross_entropy(netR(xb.to(DEVICE)), yb.to(DEVICE)).backward(); optR.step()
_, accR, _ = evaluate(netR, mk(teR))
print(f"\nrandom-chip split test accuracy {accR:.3f}  vs  scene split {test_acc:.3f}  -> gap {100 * (accR - test_acc):+.1f} points (leakage; report the scene-split number)")

In [ ]:
# the mistakes: highest-confidence false alarms and the missed ships
Xte = X[te]; yte = y[te]
fp = np.flatnonzero((pred == 1) & (yte == 0)); fn = np.flatnonzero((pred == 0) & (yte == 1))
fp = fp[np.argsort(-p_test[fp])][:8]; fn = fn[np.argsort(p_test[fn])][:8]
fig, ax = plt.subplots(2, 8, figsize=(14, 3.8))
for a, i in zip(ax[0], fp): a.imshow(Xte[i]); a.set_title(f"FP p={p_test[i]:.2f}", fontsize=8, color="red"); a.axis("off")
for a, i in zip(ax[1], fn): a.imshow(Xte[i]); a.set_title(f"FN p={p_test[i]:.2f}", fontsize=8, color="orange"); a.axis("off")
for a in ax.ravel(): a.axis("off")
plt.suptitle("What the network gets wrong: false alarms (top) and missed ships (bottom) - wakes, docks, partial hulls", y=1.02); plt.tight_layout(); plt.show()

## 7 · What did it learn?  *(Week 3, slide 'Convolution')*
The sixteen 3 × 3 × 3 kernels of the first layer are the numbers you typed by hand in Week 3 — now fitted to 2 800 chips. Then the feature maps of one ship chip after each block.

In [ ]:
W = net.backbone[0].weight.detach().cpu().numpy()            # (16, 3, 3, 3)
fig, ax = plt.subplots(2, 8, figsize=(12, 3.2))
for k, a in enumerate(ax.ravel()):
    w = W[k].transpose(1, 2, 0); w = (w - w.min()) / (w.max() - w.min() + 1e-9); a.imshow(w, interpolation="nearest"); a.axis("off"); a.set_title(f"k{k}", fontsize=8)
plt.suptitle("layer-1 kernels (RGB): edge-, colour- and brightness-detectors, learned not typed", y=1.02); plt.tight_layout(); plt.show()

i = np.flatnonzero(yte == 1)[0]; xi = to_tensor(Xte[i:i+1]).to(DEVICE)
acts = []; h_ = xi
with torch.no_grad():
    for layer in net.backbone:
        h_ = layer(h_)
        if isinstance(layer, nn.MaxPool2d): acts.append(h_[0].cpu().numpy())
fig, ax = plt.subplots(1, 1 + 3 * 4, figsize=(18, 2.2)); ax[0].imshow(Xte[i]); ax[0].set_title("input", fontsize=8); ax[0].axis("off")
for b, A in enumerate(acts):
    for j in range(4):
        a = ax[1 + b * 4 + j]; a.imshow(A[j], cmap="magma"); a.axis("off"); a.set_title(f"block {b+1} map {j} {A.shape[1]}x{A.shape[2]}", fontsize=7)
plt.suptitle("feature maps: 40x40 edges -> 20x20 parts -> 10x10 'ship-ness'", y=1.1); plt.tight_layout(); plt.show()

## 8 · From classifier to detector — the sliding window  *(slide 9: 'Detection before deep learning')*
Slide an 80 × 80 window across a whole Planet scene with a stride of 20 px, classify every window, keep the windows with p > T, merge neighbours. This is how detection was done before YOLO. Watch the clock.

In [ ]:
from scipy import ndimage
scene_img = np.array(Image.open(SCENE_SFBAY_1).convert("RGB")); H, W_ = scene_img.shape[:2]
STRIDE, WIN, T = 20, 80, 0.5
rows = range(0, H - WIN + 1, STRIDE); cols = range(0, W_ - WIN + 1, STRIDE)
windows = np.stack([scene_img[r:r+WIN, c:c+WIN] for r in rows for c in cols]); print("scene", scene_img.shape, "->", len(windows), "windows")
t0 = time.time(); probs = []
net.eval()
with torch.no_grad():
    for k in range(0, len(windows), 512): probs.append(F.softmax(net(to_tensor(windows[k:k+512]).to(DEVICE)), 1)[:, 1].cpu())
probs = torch.cat(probs).numpy().reshape(len(rows), len(cols)); dt = time.time() - t0
heat = probs > T; lab, nblob = ndimage.label(heat); boxes = []
for b in range(1, nblob + 1):
    rr, cc = np.where(lab == b); r0, c0 = rows[rr.min()], cols[cc.min()]; r1, c1 = rows[rr.max()] + WIN, cols[cc.max()] + WIN; boxes.append((c0, r0, c1 - c0, r1 - r0, probs[rr, cc].max()))
print(f"{len(windows)} windows classified in {dt:.1f} s on {DEVICE}; {int(heat.sum())} windows above T={T} -> {len(boxes)} merged boxes")
fig, ax = plt.subplots(1, 2, figsize=(18, 7))
ax[0].imshow(scene_img); ax[0].set_title(f"{len(boxes)} detections (p > {T}) - sliding window, stride {STRIDE}")
for (c0, r0, w, h, p) in boxes: ax[0].add_patch(plt.Rectangle((c0, r0), w, h, fill=False, ec="lime", lw=1.5))
ax[0].axis("off"); ax[1].imshow(probs, cmap="magma", vmin=0, vmax=1); ax[1].set_title("ship probability per window (the 'heat map')"); ax[1].axis("off")
plt.tight_layout(); plt.show()

**Look at the result.** Some boxes are ships; some are docks, wakes or piers the training set never showed; one box may cover two adjacent ships; every box is a multiple of 80 px whatever the ship's size and heading. Now change `T` to 0.9 and to 0.2, and `STRIDE` to 10 — count boxes and seconds each time.

## 9 · Four questions  *(hand in with Lab 2)*
1. **Split.** Report test accuracy, precision and recall for the scene split, and the random-split accuracy. Explain the gap in two sentences and say which number belongs on a dataset card.
2. **Baseline.** Did the CNN beat the random forest on the test scenes? By how much, and on which metric? If the margin is small, what does that say about this task at 3 m GSD?
3. **Threshold.** From the sweep, pick a threshold for (a) a monthly harbour statistics report and (b) a coast-guard alert. Give the precision and recall at each and one sentence on who pays for the errors.
4. **Sliding window vs YOLO.** From section 8: how many windows, how many seconds, how many boxes, and three things a box could not tell you (size? heading? two ships?). In Lab 2 you will run YOLO26 on a 1024-px image in ~30 ms — name the two design ideas that make that possible (slides 12, 16).

✏️ **YOUR ANSWERS**

1. 

2. 

3. 

4. 

---
*Data: 'Ships in Satellite Imagery' (Planet PlanetScope chips, San Francisco Bay and Long Beach, 3 m, CC-BY-SA 4.0), repackaged as `data/shipsnet/shipsnet_80x80.npz` with the scene id of every chip. Course repo: [trongan93/dl-space-2026](https://github.com/trongan93/dl-space-2026).*